In [ ]:
import pyodbc
import csv

# -----------------------------------
# 1. DB 연결
# -----------------------------------
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=DBPr01;"
    "UID=testuser;"
    "PWD=1234;"
    "TrustServerCertificate=yes;"
)

cur = conn.cursor()

# -----------------------------------
# 2. CSV 파일 경로 / 테이블 이름
# -----------------------------------
fn = r'/Users/dh/projects/Lectures/Dataenginering/703cdf01-ff09-4b86-b017-6e8d87b11fd2.csv'
table_name = 'ga0_raw'

# -----------------------------------
# 3. CSV header 읽기
# -----------------------------------
with open(fn, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)

print("CSV header count:", len(header))
print("CSV header:", header)

# -----------------------------------
# 4. 컬럼명 정리
#    공백 -> _
#    소문자 변환
# -----------------------------------
clean_header = []
for col in header:
    col = col.strip().lower().replace(' ', '_')
    clean_header.append(col)

print("Clean header:", clean_header)

# -----------------------------------
# 5. 기존 테이블 삭제
# -----------------------------------
cur.execute(f"""
IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
    DROP TABLE {table_name};
""")
conn.commit()

# -----------------------------------
# 6. CREATE TABLE 자동 생성
#    모든 컬럼을 일단 VARCHAR(300)으로 생성
# -----------------------------------
column_defs = []
for col in clean_header:
    column_defs.append(f"[{col}] VARCHAR(300)")

create_sql = f"""
CREATE TABLE {table_name} (
    id_column INT NOT NULL IDENTITY(1,1) PRIMARY KEY,
    {', '.join(column_defs)}
);
"""

print("=== CREATE SQL ===")
print(create_sql)

cur.execute(create_sql)
conn.commit()

# -----------------------------------
# 7. INSERT SQL 자동 생성
# -----------------------------------
column_names_sql = ", ".join([f"[{col}]" for col in clean_header])
placeholders = ", ".join(["?"] * len(clean_header))

insert_sql = f"""
INSERT INTO {table_name} (
    {column_names_sql}
) VALUES (
    {placeholders}
);
"""

print("=== INSERT SQL ===")
print(insert_sql)

# -----------------------------------
# 8. CSV 데이터 읽기
# -----------------------------------
data_rows = []

with open(fn, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)  # header skip

    for row in reader:
        data_rows.append(tuple(row))

print("Rows to insert:", len(data_rows))

# -----------------------------------
# 9. executemany로 한꺼번에 insert
# -----------------------------------
cur.fast_executemany = True
cur.executemany(insert_sql, data_rows)
conn.commit()

print("INSERT 완료")

# -----------------------------------
# 10. 확인
# -----------------------------------
cur.execute(f"SELECT COUNT(*) FROM {table_name};")
print("Rows in table:", cur.fetchone()[0])

# -----------------------------------
# 11. 종료
# -----------------------------------
cur.close()
conn.close()

 MySQL용 코드 → MS-SQL Server용 코드가 됩니다.

핵심 변경점은 4개입니다.
	•	pymysql → pyodbc
	•	AUTO_INCREMENT → IDENTITY(1,1)
	•	MySQL의 백틱 `col` → SQL Server의 대괄호 [col]
	•	연결 문자열 방식 변경

⸻

바뀐 부분만 짧게 설명

1. 라이브러리

MySQL:

import pymysql

SQL Server:

import pyodbc


⸻

2. 연결 방식

MySQL은:

pymysql.connect(...)

SQL Server는:

pyodbc.connect("DRIVER=...;SERVER=...;DATABASE=...;UID=...;PWD=...")


⸻

3. 자동 증가

MySQL:

AUTO_INCREMENT

SQL Server:

IDENTITY(1,1)


⸻

4. 컬럼 표시

MySQL:

`column_name`

SQL Server:

[column_name]


⸻

5. placeholder

MySQL pymysql:

%s

SQL Server pyodbc:

?

이게 아주 중요합니다.

즉, 이 부분:

placeholders = ", ".join(["?"] * len(clean_header))

처럼 바꿔야 합니다.

⸻

추가로 주의할 점

ODBC Driver 설치 필요

맥에서 SQL Server에 연결하려면 보통 ODBC Driver 17 for SQL Server 또는 18이 설치되어 있어야 합니다.

예를 들어 에러가 나면 이런 경우가 많습니다.
	•	드라이버 미설치
	•	서버 이름 다름
	•	SQL Server 인증 설정 문제

⸻

만약 서버 이름이 다르면

지금은 예시로:

SERVER=localhost;

를 썼습니다.

SQL Server가 다른 이름이면 바꿔야 합니다.

예:

SERVER=localhost,1433;

또는

SERVER=192.168.0.10;

또는 Windows SQL Server면

SERVER=localhost\SQLEXPRESS;

하지만 맥에서 pyodbc로 접속할 때는 보통 서버주소,포트 형식이 더 안정적입니다.

⸻

한 줄 핵심

이 코드를 SQL Server로 바꾸는 핵심은:
	•	pyodbc
	•	IDENTITY(1,1)
	•	[col]
	•	?

입니다.

원하시면 다음 답변에서 제가 이 코드를 MS-SQL용으로 한 줄씩 주석 달아서 다시 풀어드리겠습니다.